# 2.9 Model Context Protocol: servers, tools, resources, prompts, client

Step 2, lesson 9 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287780) (Anthropic Academy).
Exam domain: **2 Tool Design & MCP Integration** (18%). Step 7 of the path (*Introduction to Model Context Protocol*) goes deeper; this is the first pass.

What this notebook covers, in the course's order:

1. What MCP is: hosts, clients, servers and transports
2. The three primitives: tools, resources, prompts, and who controls each
3. The course project: a document server ([`mcp_docs_server/server.py`](mcp_docs_server/server.py))
4. A client: connect over stdio, list and call everything
5. Resources in an app: `@` mentions
6. Prompts and a Claude tool loop over MCP tools
7. Testing a server with the MCP Inspector

> **Model used here:** `claude-sonnet-5-5`, for about 4 requests in sections 5 and 6. Everything else talks only to the local server.
>
> **Exam scope.** Building and configuring local servers like this one is in scope. Hosting or deploying MCP servers (remote HTTP servers, auth, scaling) is on the exam guide's out-of-scope list.

## Setup

In [1]:
import json
import re
import sys
import textwrap
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
client = Anthropic()            # the Anthropic client; MCP connections are called `docs` below
model = "claude-sonnet-5-5"
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)


def text_of(message):
    """Return the text of the first text block in a response."""
    return next(block.text for block in message.content if block.type == "text")


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

## 1. What MCP is

Without a standard, every AI application writes its own glue for every service: a GitHub tool for app A, another GitHub tool for app B, and so on. The **Model Context Protocol** (MCP) is an open standard for that glue. A service is wrapped once, as an **MCP server**, and any MCP-capable application can use it.

| Role | What it is | In this notebook |
|---|---|---|
| **Host** | The AI application the user works in | this notebook (elsewhere: Claude Code, Claude Desktop) |
| **Client** | A connection inside the host, one per server | the `Client` object below |
| **Server** | A program that exposes capabilities for one service | `mcp_docs_server/server.py` |

They talk JSON-RPC (`tools/list`, `tools/call`, `resources/read`, `prompts/get`, ...) over a **transport**:

- **stdio**: the host starts the server as a local subprocess and talks over its stdin/stdout. That's what this notebook and most Claude Code setups use.
- **Streamable HTTP**: the server runs remotely at a URL.

MCP doesn't replace tool use. Claude still receives ordinary tool definitions and returns `tool_use` blocks; MCP only standardises **where the definitions come from and who runs the tool**. Claude never talks to the server; the host does, on Claude's behalf.

## 2. Three primitives, three controllers

| Primitive | Controlled by | What it is | In the document server |
|---|---|---|---|
| **Tools** | the **model** | Actions or queries Claude decides to call | `read_doc_contents`, `edit_document` |
| **Resources** | the **application** | Read-only data, addressed by a URI, that the app chooses to put into the context | `docs://documents` (the list), `docs://documents/{doc_id}` (a URI template for one document) |
| **Prompts** | the **user** | Ready-made, tested instructions that a user picks, for example as a slash command | `format`, `summarize` |

A useful test when designing a server: *who decides this should happen?* If Claude should decide mid-task, it's a tool. If the app should load it up front (an attached file, a mentioned document), it's a resource. If a person chooses a workflow by name, it's a prompt.

## 3. The server

> **Changed since the course.** The course installs `mcp[cli]` 1.x with `uv` and builds the server with `FastMCP`. The installed SDK is **mcp 2.x**, where:
> - `FastMCP` is now **`MCPServer`** (`from mcp.server.mcpserver import MCPServer`); the decorators `@mcp.tool()`, `@mcp.resource(uri)` and `@mcp.prompt()` work as before.
> - The client side has a ready-made **`Client`** class. The course writes its own `MCPClient` wrapper around `ClientSession` and `stdio_client`; `Client` does that job.
> - A tool that raises an ordinary exception now returns only a generic "Error executing tool ..." to the model (the details stay in the server log). To give Claude a message it can act on, raise **`ToolError`**.
> - Client and server negotiate the protocol version; with 2.x on both sides it's the 2026-07-28 revision.

The old import fails with a pointer to the migration guide:

In [2]:
try:
    from mcp.server.fastmcp import FastMCP
except ModuleNotFoundError as e:
    print(textwrap.fill(str(e), 110))

No module named 'mcp.server.fastmcp'. This is mcp 2.x, where FastMCP was renamed to MCPServer (from
mcp.server.mcpserver import MCPServer) and other APIs changed; see the migration guide at
https://py.sdk.modelcontextprotocol.io/v2/migration/#fastmcp-renamed-to-mcpserver or pin 'mcp<2' to keep
running v1 code.


Here is the whole server (also in [`mcp_docs_server/server.py`](mcp_docs_server/server.py)). Things to notice:

- The **function name, type hints and docstring-like `description`** become the tool definition Claude sees. `Annotated[str, Field(description=...)]` puts a description on each parameter of the JSON schema.
- The tool descriptions say **when** to use the tool and **what makes it fail**, not just what it does.
- The documents are a Python dict, so edits last as long as the server process.

In [3]:
print(Path("mcp_docs_server/server.py").read_text())

"""A small MCP server that exposes an in-memory set of documents (lesson 2.9).

Primitives, in MCP terms:
- tools (model-controlled):     read_doc_contents, edit_document
- resources (app-controlled):   docs://documents (the list of ids), docs://documents/{doc_id} (one document)
- prompts (user-controlled):    format (rewrite a document in Markdown), summarize

The documents live in a Python dict, so edits last only as long as the server process.
Built with the `mcp` Python SDK 2.x, where the v1 `FastMCP` class is now `MCPServer`.

Run it over stdio (what Claude Code and the notebook client do):
    python server.py
"""

from typing import Annotated

from pydantic import Field

from mcp.server.mcpserver import MCPServer, UserMessage
from mcp.server.mcpserver.exceptions import ResourceError, ToolError

mcp = MCPServer(
    "docs",
    instructions="Read and edit a small set of internal documents. Document ids look like file names, e.g. 'roadmap.md'.",
)

docs: dict[str, str] = {
    "ro

## 4. A client

`StdioServerParameters` says how to start the server: here, the notebook's own Python running `server.py`. `Client(...)` used with `async with` starts the process, negotiates the protocol, and stops the process when the block ends. **Every `async with` block is a fresh server process**, so edits made in one block are gone in the next.

Jupyter allows `await` at the top level of a cell, so no `asyncio.run` is needed. The server's log (stderr) goes to a file in `outputs/` to keep the notebook tidy.

In [4]:
server_params = StdioServerParameters(command=sys.executable, args=["mcp_docs_server/server.py"])
server_log = open(OUT / "mcp_server.log", "w")


def connect():
    """A new client connection; each `async with connect()` starts a fresh server process."""
    return Client(stdio_client(server_params, errlog=server_log))

Listing what the server offers. These four `list` calls are what a host does when it connects: tools go to the model, resources and prompts to the app's UI.

In [5]:
async with connect() as docs:
    print("protocol:", docs.protocol_version, "| server:", getattr(docs.server_info, "name", "?"))
    print("\nTOOLS")
    for tool in (await docs.list_tools()).tools:
        print(f"  {tool.name}({', '.join(tool.input_schema['properties'])}): {textwrap.shorten(tool.description, 90)}")
    print("\nRESOURCES")
    for resource in (await docs.list_resources()).resources:
        print(f"  {resource.uri} ({resource.mime_type}): {resource.description}")
    for template in (await docs.list_resource_templates()).resource_templates:
        print(f"  {template.uri_template} ({template.mime_type}): {template.description}  [template]")
    print("\nPROMPTS")
    for prompt in (await docs.list_prompts()).prompts:
        print(f"  {prompt.name}({', '.join(a.name for a in prompt.arguments)}): {prompt.description}")

protocol: 2026-07-28 | server: docs

TOOLS
  read_doc_contents(doc_id): Read the full text of one document. Use it before answering a question about a [...]
  edit_document(doc_id, old_str, new_str): Edit a document by replacing an exact piece of text with new text. Read the document [...]

RESOURCES
  docs://documents (application/json): The ids of all documents
  docs://documents/{doc_id} (text/plain): The text of one document  [template]

PROMPTS
  format(doc_id): Rewrite a document in Markdown and save it with the edit tool.
  summarize(doc_id): Summarise a document in one sentence.


Calling tools and reading resources directly, without Claude. Note the three outcomes of a tool call:

- success: `is_error` is false; the text is in `content` (and, because the function returns `str`, also in `structured_content`);
- a failure the tool anticipated (`ToolError`): `is_error` is true and the message tells the caller how to recover (here: the list of valid ids);
- the edit, checked afterwards by reading the resource, in the **same** connection (the same server process).

In [6]:
async with connect() as docs:
    result = await docs.call_tool("read_doc_contents", {"doc_id": "budget.pdf"})
    print("read:   is_error =", result.is_error, "|", result.content[0].text)

    result = await docs.call_tool("read_doc_contents", {"doc_id": "budget.docx"})
    print("\nwrong id: is_error =", result.is_error, "|", result.content[0].text)

    result = await docs.call_tool("edit_document", {"doc_id": "budget.pdf", "old_str": "1.2 million", "new_str": "1.3 million"})
    print("\nedit:   is_error =", result.is_error, "|", result.content[0].text)
    resource = await docs.read_resource("docs://documents/budget.pdf")
    print("check:  ", resource.contents[0].text[:60], "...")

read:   is_error = False | The 2027 infrastructure budget is 1.2 million euros, 8 percent more than 2026. Most of the increase pays for GPU capacity for the recommendation models.

wrong id: is_error = True | Error executing tool read_doc_contents: Document 'budget.docx' not found. Known ids: api-guidelines.md, budget.pdf, incident-0312.txt, onboarding.docx, retro-notes.txt, roadmap.md

edit:   is_error = False | Edited 'budget.pdf'.
check:   The 2027 infrastructure budget is 1.3 million euros, 8 perce ...


**Exam angle (Domain 2).** A tool error should be an `is_error` result with a message that says *what went wrong and what to do next* ("not found; known ids are ..."), so the agent can recover on its own. A bare "error" or an empty result makes Claude guess, retry blindly, or report success. It also matters to distinguish an error ("couldn't check") from a valid empty answer ("checked; nothing found").

## 5. Resources in an app: `@` mentions

The course's chat app lets the user type `@` and pick a document. That's a **resource**: the *application* reads it and puts its text in the prompt, so Claude has it from the start without spending a tool call. The app first reads `docs://documents` to know which ids exist (that list could also drive an autocomplete menu).

In [7]:
async def expand_mentions(docs, text):
    """Replace @doc_id mentions with the document text, read from the server's resources."""
    known = json.loads((await docs.read_resource("docs://documents")).contents[0].text)
    blocks = []
    for doc_id in re.findall(r"@([\w.-]+)", text):
        if doc_id in known:
            content = (await docs.read_resource(f"docs://documents/{doc_id}")).contents[0].text
            blocks.append(f'<document id="{doc_id}">\n{content}\n</document>')
    return "\n".join(blocks) + f"\n\n{text}"


async with connect() as docs:
    prompt = await expand_mentions(docs, "What caused the outage in @incident-0312.txt and what stops it happening again")
print(prompt, "\n")

response = client.messages.create(
    model=model, max_tokens=300, output_config={"effort": "low"},
    messages=[{"role": "user", "content": prompt}],
)
print("Claude:", text_of(response))

<document id="incident-0312.txt">
Incident 0312. On 12 March the search API returned errors for 47 minutes. Cause: an expired TLS certificate on the internal index cluster. Fix: certificate renewed, expiry alerts added at 30 and 7 days.
</document>

What caused the outage in @incident-0312.txt and what stops it happening again 



Claude: **Cause:** An expired TLS certificate on the internal index cluster. This caused the search API to return errors for 47 minutes on 12 March.

**Fix and prevention:**
- The certificate was renewed, which resolved the outage.
- Expiry alerts were added at 30 days and 7 days before expiration, so the team gets advance warning before a certificate lapses again.

Note that the prevention is alerting-based rather than automated renewal, so it still relies on someone acting on the alerts in time.


## 6. Prompts, and Claude using MCP tools

To let Claude use the server's tools, the host converts each MCP tool into an Anthropic tool definition (they map one to one: name, description, JSON schema), runs the usual tool loop, and forwards every `tool_use` to `call_tool` on the server. The MCP `is_error` flag becomes the `tool_result`'s `is_error`.

In [8]:
def to_anthropic_tool(tool):
    return {"name": tool.name, "description": tool.description, "input_schema": tool.input_schema}


async def run_with_mcp_tools(docs, messages, max_turns=6):
    tools = [to_anthropic_tool(t) for t in (await docs.list_tools()).tools]
    for turn in range(max_turns):
        response = client.messages.create(model=model, max_tokens=2000, tools=tools, messages=messages)
        add_assistant_message(messages, response.content)
        if response.stop_reason != "tool_use":
            return text_of(response)
        results = []
        for block in response.content:
            if block.type == "tool_use":
                result = await docs.call_tool(block.name, block.input)
                text = "\n".join(c.text for c in result.content if c.type == "text")
                print(f"  turn {turn + 1}: {block.name}({textwrap.shorten(json.dumps(block.input), 70)})\n"
                      f"          -> {'ERROR: ' if result.is_error else ''}{textwrap.shorten(text, 70)}")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": text, "is_error": bool(result.is_error)})
        add_user_message(messages, results)
    return "Stopped: too many turns."

Now the **user-controlled** path: the user picks the `format` prompt (in the course's app, by typing `/format`) and a document. The host fetches the prompt's messages from the server, sends them to Claude with the tools, and Claude does the reading and editing.

In [9]:
async with connect() as docs:
    prompt = await docs.get_prompt("format", {"doc_id": "retro-notes.txt"})
    messages = [{"role": m.role, "content": m.content.text} for m in prompt.messages]
    print("prompt from the server:\n" + textwrap.indent(messages[0]["content"], "  | ") + "\n")

    final = await run_with_mcp_tools(docs, messages)
    print("\nClaude's reply:\n" + final)

    stored = (await docs.read_resource("docs://documents/retro-notes.txt")).contents[0].text
    print("\nstored document starts with:", repr(stored[:80]))

prompt from the server:
  | Reformat the document <document_id>retro-notes.txt</document_id> as Markdown.

  | Use headings, bullet lists and bold text where they make it easier to scan. Keep every fact; don't add new ones.
  | Read the document with the read_doc_contents tool, then save the new version with the edit_document tool
  | (replace the whole old text with the Markdown text). Finally, reply with the Markdown version only.



  turn 1: read_doc_contents({"doc_id": "retro-notes.txt"})
          -> Sprint 41 retro. Went well: releases on time, fewer flaky tests. [...]


  turn 2: edit_document({"doc_id": "retro-notes.txt", "old_str": "Sprint 41 retro. Went [...])
          -> Edited 'retro-notes.txt'.



Claude's reply:
# Sprint 41 Retro

## Went well

- Releases on time
- Fewer flaky tests

## To improve

- Code reviews wait too long, with an average of **26 hours**

## Action

- Two reviewers on rotation each day

stored document starts with: '# Sprint 41 Retro\n\n## Went well\n\n- Releases on time\n- Fewer flaky tests\n\n## To i'


The prompt's wording was written once by the server's author, who knows the documents best; the user only chose it. Claude then decided which tools to call (model-controlled), and the app could read the result back as a resource (application-controlled): all three primitives in one flow.

> **Beyond the course.** Two shortcuts exist for what this section wires by hand:
> - The Anthropic Python SDK has MCP helpers (`anthropic.lib.tools.mcp`, beta, installed with `anthropic[mcp]`) that convert MCP tools, prompts and resources for the SDK's tool runner.
> - The Messages API has an **MCP connector** (`mcp_servers` parameter, beta) that lets the API call a *remote* MCP server directly. It doesn't help with a local stdio server like this one, and hosting remote servers is out of exam scope.

## 7. Testing a server with the MCP Inspector

The **MCP Inspector** is a browser UI for poking at a server by hand: list and call tools, read resources, render prompts, see the raw JSON-RPC messages. Run it from the repo root (Node.js is in `~/.local/node/bin` on this machine):

```bash
PATH="$HOME/.local/node/bin:$PATH" npx @modelcontextprotocol/inspector \
    .venv/bin/python 02-building-with-the-claude-api/mcp_docs_server/server.py
```

The course starts it with `mcp dev mcp_server.py`, which wraps the same command (and expects `uv`). Testing a server in the Inspector before connecting it to Claude separates "the server is broken" from "Claude is using it badly".

**Exam angle (Domain 2).** Points from this lesson that the guide's scenarios lean on:

- Tool **descriptions** drive tool choice. If Claude picks the wrong tool or misuses one, improve the descriptions (when to use it, input format, examples, what it can't do) before adding routing logic or more tools.
- Keep each agent's tool set **small and relevant**; many overlapping tools make selection worse.
- Return **structured, actionable errors** with `is_error`, and keep "failed" distinct from "nothing found".
- **Resources** give Claude context up front without a tool call; **prompts** package a workflow for users.
- Where servers are configured (project `.mcp.json` vs user scope) and how credentials stay out of the repo (`${VAR}` expansion) is Claude Code configuration: next lesson.

## Recap

- MCP standardises how a host's clients discover and call capabilities on servers, over stdio (local) or Streamable HTTP (remote).
- Tools are model-controlled, resources application-controlled, prompts user-controlled.
- mcp 2.x: `MCPServer` (formerly `FastMCP`), a ready-made `Client`, `ToolError` for messages the model should see.
- Using MCP tools with Claude = convert definitions, run the normal tool loop, forward each `tool_use` to `call_tool`.

Next: [2.10 Claude Code and MCP](10-claude-code-and-mcp.ipynb).